In [1]:
# ============================================================
# BVMT News Scraper — ilboursa.com
# File: notebooks/scrape_ilboursa.py
# Copy each section into a separate Jupyter cell
# ============================================================
#
# WHAT IS WEB SCRAPING?
# ---------------------
# A website is just HTML text. When you open a browser, the browser
# downloads that HTML and renders it visually. Scraping means:
#   1. We download the same HTML using Python (requests library)
#   2. We parse it using BeautifulSoup to find specific elements
#   3. We extract the text/links we want and save them to our DB
#
# The challenge: websites don't want bots scraping them.
# They check your request headers. If it looks like a bot, they
# return a 403 error (forbidden). We fix this by pretending to
# be a real browser.
# ============================================================
 

In [2]:
import requests          # sends HTTP requests (like a browser fetching a page)
from bs4 import BeautifulSoup  # parses HTML and lets us search for elements by tag/class
import psycopg2          # connects Python to PostgreSQL
import psycopg2.extras   # extra utilities: RealDictCursor (rows as dicts), execute_batch
import pandas as pd      # for displaying results nicely
from dotenv import load_dotenv
from datetime import datetime, date
import time              # for time.sleep() — we pause between requests to be polite
import os
import re                # regular expressions — for cleaning text patterns
 
load_dotenv()
 
# DB connection helper — same pattern as all the other notebooks
def get_conn():
    return psycopg2.connect(
        host=os.getenv('DB_HOST'),
        port=int(os.getenv('DB_PORT', 5432)),
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD')
    )
 
print("Imports OK")
 

Imports OK


In [13]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — FINAL CORRECTED ticker mapping                 ║
# ║  Replace your old Cell 2 entirely with this              ║
# ╚══════════════════════════════════════════════════════════╝
#
# All 26 corrections from your manual verification applied.
# 3 special cases explained below.
#
# SPECIAL CASE 1 — MODERN LEASING:
#   In 2019 Modern Leasing renamed to BH Leasing.
#   On ilboursa both names share the code BHL.
#   Your DB has the old name 'MODERN LEASING' in company_metadata
#   so we map it to BHL — it will scrape BH Leasing news which
#   is the same company. This is correct behaviour.
#
# SPECIAL CASE 2 — BEST LEASE:
#   Your code you confirmed was BHL — but that is BH Leasing.
#   BEST LEASE has its own separate code. Based on search results
#   the correct code is 'BEST' — marked TODO below for you to
#   verify by visiting: https://www.ilboursa.com/marches/news_valeur?s=BEST
#
# SPECIAL CASE 3 — SITEX:
#   Not found on ilboursa. Likely very rarely covered or suspended.
#   Excluded from the mapping — it will simply have 0 articles.

ILBOURSA_TICKERS = {
    # ── Banking ──────────────────────────────────────────────────────────
    'AMEN BANK':         'AB',       # CONFIRMED by you
    'BIAT':              'BIAT',     # CONFIRMED by you
    'ATTIJARI BANK':     'TJARI',    # CONFIRMED by you
    'BH':                'BH',       # CONFIRMED by you
    'BNA':               'BNA',
    'BT':                'BT',
    'STB':               'STB',
    'UIB':               'UIB',
    'ATB':               'ATB',
    'UBCI':              'UBCI',
    'WIFACK INT BANK':   'WIFAK',    # CORRECTED: was WIFACK, now WIFAK

    # ── Insurance ────────────────────────────────────────────────────────
    'STAR':              'STAR',
    'ASTREE':            'AST',      # CORRECTED: was ASTREE, now AST
    'BH ASSURANCE':      'BHASS',
    'TUNIS RE':          'TRE',      # CORRECTED: was 'TUNIS RE' (space), now TRE

    # ── Leasing / Finance ─────────────────────────────────────────────────
    'TUNISIE LEASING F': 'TLS',      # CORRECTED: was TL, now TLS
    'ATTIJARI LEASING':  'TJL',      # CORRECTED: was ATJL, now TJL
    'HANNIBAL LEASE':    'HL',       # CORRECTED: was HNBL, now HL
    'BEST LEASE':        'BL',     # ⚠ VERIFY: visit ?s=BEST to confirm
    'MODERN LEASING':    'BHL',      # See SPECIAL CASE 1 above
    'TUNISIE VALEURS':   'TVAL',     # CORRECTED: was TV, now TVAL
    'SPDIT - SICAF':     'SPDIT',
    'PLAC. TSIE-SICAF':  'PLTU',     # CORRECTED: was PLTS, now PLTU
    'TUNINVEST-SICAR':   'TINV',
    'BTE (ADP)':         'BTE',
    'SIMPAR':            'SIMPA',    # CORRECTED: was SIMPAR, now SIMPA

    # ── Industry / Food / Beverage ────────────────────────────────────────
    'SFBT':              'SFBT',     # CONFIRMED by you
    'DELICE HOLDING':    'DH',       # CORRECTED: was DLICE, now DH
    'POULINA GP HOLDING':'PGH',      # CONFIRMED by you
    'ALKIMIA':           'ALKIM',    # CORRECTED: was ALKIMIA, now ALKIM
    'ADWYA':             'ADWYA',
    'SITS':              'SITS',
    'SOTIPAPIER':        'STPAP',
    'SOMOCER':           'SOMOC',    # CORRECTED: was SOMOCER, now SOMOC
    'SOTUMAG':           'MGR',      # CORRECTED: was SOTUMAG, now MGR
    'SOTUVER':           'SOTUV',    # CORRECTED: was SOTUVER, now SOTUV
    'SOTRAPIL':          'STPIL',
    'SIPHAT':            'SIPHA',    # CORRECTED: was SIPHAT, now SIPHA
    'UNIMED':            'UMED',     # CORRECTED: was UNIMED, now UMED
    'MAGASIN GENERAL':   'MAG',      # CORRECTED: was MG, now MAG
    'MONOPRIX':          'MNP',      # CORRECTED: was MONOPRIX, now MNP

    # ── Technology / Telecom ─────────────────────────────────────────────
    'ONE TECH HOLDING':  'OTH',      # CORRECTED: was OTECH, now OTH
    'SOTETEL':           'SOTET',    # CORRECTED: was SOTETEL, now SOTET
    'CELLCOM':           'CELL',     # CORRECTED: was CELLCOM, now CELL
    'TELNET HOLDING':    'TLNET',
    'GIF-FILTER':        'GIF',

    # ── Auto / Transport ──────────────────────────────────────────────────
    'ENNAKL AUTOMOBILES':'NAKL',     # CORRECTED: was ENNAKL, now NAKL
    'CITY CARS':         'CC',
    'EURO-CYCLES':       'ECYCL',    # CORRECTED: was ECY, now ECYCL
    'TUNISAIR':          'TAIR',     # CORRECTED: was TUNAIR, now TAIR
    'ARTES':             'ARTES',

    # ── Construction / Materials ──────────────────────────────────────────
    'CIMENTS DE BIZERTE':'SCB',
    'ELBENE INDUSTRIE':  'ELBEN',    # CORRECTED: was ELBENE, now ELBEN
    # SITEX excluded — not found on ilboursa
    'SAH':               'SAH',
    'SIAME':             'SIAME',
    'ELECTROSTAR':       'LSTR',     # CORRECTED: was ESTAR, now LSTR
    'ASSAD':             'ASSAD',

    # ── Other ─────────────────────────────────────────────────────────────
    'ICF':               'ICF',
    'CIL':               'CIL',
    'ATL':               'ATL',
    'TPR':               'TPR',
    'SOPAT':             'SOPAT',
    'MPBS':              'MPBS',
    'UADH':              'UADH',
    'STEQ':              'STEQ',
    'AIR LIQUDE TSIE':   'AL',       # CORRECTED: was AIRLT, now AL
    'ATELIER MEUBLE INT':'SAM',      # CORRECTED: was AMI, now SAM (real ticker)
    'ESSOUKNA':          'SOKNA',    # CORRECTED: was ESSOUKNA, now SOKNA
}

print(f"Ticker mapping loaded: {len(ILBOURSA_TICKERS)} stocks")
print()

# Quick sanity check — print all entries sorted by code
print(f"{'DB Ticker':<30} {'ilboursa code':<12} {'Notes'}")
print("-" * 70)
NOTES = {
    'MODERN LEASING':    'renamed to BH Leasing in 2019',
    'BEST LEASE':        '⚠ VERIFY at ?s=BEST',
    'WIFACK INT BANK':   'corrected',
    'TUNISIE LEASING F': 'corrected',
}
for ticker, code in ILBOURSA_TICKERS.items():
    note = NOTES.get(ticker, '')
    print(f"{ticker:<30} {code:<12} {note}")

Ticker mapping loaded: 68 stocks

DB Ticker                      ilboursa code Notes
----------------------------------------------------------------------
AMEN BANK                      AB           
BIAT                           BIAT         
ATTIJARI BANK                  TJARI        
BH                             BH           
BNA                            BNA          
BT                             BT           
STB                            STB          
UIB                            UIB          
ATB                            ATB          
UBCI                           UBCI         
WIFACK INT BANK                WIFAK        corrected
STAR                           STAR         
ASTREE                         AST          
BH ASSURANCE                   BHASS        
TUNIS RE                       TRE          
TUNISIE LEASING F              TLS          corrected
ATTIJARI LEASING               TJL          
HANNIBAL LEASE                 HL           
BEST LEASE      

In [15]:
# ╔══════════════════════════════════════════════════════════╗
# ║  The browser headers (critical to avoid 403)            ║
# ╚══════════════════════════════════════════════════════════╝
#
# WHY DO WE NEED HEADERS?
# When your browser visits a website, it sends "headers" — extra
# information that identifies who is making the request.
#
# NOTE ABOUT COMPRESSION:
# We intentionally avoid 'br' (brotli) in Accept-Encoding because
# some environments receive undecoded binary payloads with br.
# gzip/deflate is stable with requests.

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Session with browser headers                   ║
# ╚══════════════════════════════════════════════════════════╝

SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) '
        'Gecko/20100101 Firefox/124.0'
    ),
    'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'fr-TN,fr;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer':         'https://www.ilboursa.com/',
    'Accept-Encoding': 'gzip, deflate',
    'Connection':      'keep-alive',
})

# Visit the homepage first to receive cookies
try:
    resp = SESSION.get('https://www.ilboursa.com/', timeout=15)
    print(f"Homepage: HTTP {resp.status_code}")
    print(f"Cookies received: {dict(SESSION.cookies)}")
except Exception as e:
    print(f"Warning: could not reach homepage: {e}")


Homepage: HTTP 200
Cookies received: {}


In [16]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Core parser function (fixed)                  ║
# ╚══════════════════════════════════════════════════════════╝
#
# PARSING STRATEGY (flat HTML structure):
# <span class="sp1">DATE</span>
# <a href="slug_12345">TITLE</a><br>

from urllib.parse import quote, urljoin


def parse_date(date_str: str):
    """Convert '18/02/26 11:32' to a Python date object."""
    try:
        date_part = date_str.strip().split()[0]
        return datetime.strptime(date_part, '%d/%m/%y').date()
    except (ValueError, IndexError):
        try:
            date_part = date_str.strip().split()[0]
            return datetime.strptime(date_part, '%d/%m/%Y').date()
        except (ValueError, IndexError):
            return date.today()


def parse_ilboursa_page(ticker: str, ilboursa_code: str, max_articles: int = 30, debug: bool = False) -> list:
    """Fetch and parse news articles for one stock from ilboursa.com."""
    url = f'https://www.ilboursa.com/marches/news_valeur?s={ilboursa_code}'
    articles = []

    try:
        resp = SESSION.get(url, timeout=15)

        if resp.status_code == 403:
            if debug:
                print(f"  Got 403 for {ticker}, waiting 30s and retrying...")
            time.sleep(30)
            resp = SESSION.get(url, timeout=15)

        if resp.status_code != 200:
            if debug:
                print(f"  HTTP {resp.status_code} for {ticker} — skipping")
            return []

        # Decode from bytes explicitly to avoid compressed-text edge cases.
        html_text = resp.content.decode(resp.encoding or 'utf-8', errors='replace')
        soup = BeautifulSoup(html_text, 'html.parser')

        # Fallback single retry with explicit safe encoding header.
        if not soup.find('span', class_='sp1'):
            retry_headers = dict(SESSION.headers)
            retry_headers['Accept-Encoding'] = 'gzip, deflate'
            retry_resp = SESSION.get(url, timeout=15, headers=retry_headers)
            if retry_resp.status_code == 200:
                retry_html = retry_resp.content.decode(retry_resp.encoding or 'utf-8', errors='replace')
                soup = BeautifulSoup(retry_html, 'html.parser')

        date_spans = soup.find_all('span', class_='sp1')[:max_articles]

        if debug:
            print(f"  Found {len(date_spans)} date spans for {ticker}")

        for span in date_spans:
            date_text = span.get_text(strip=True)
            published = parse_date(date_text)

            a_tag = None
            cursor = span.next_sibling

            for _ in range(10):
                if cursor is None:
                    break
                if hasattr(cursor, 'name') and cursor.name == 'a':
                    a_tag = cursor
                    break
                cursor = cursor.next_sibling

            if a_tag is None:
                continue

            title = a_tag.get_text(strip=True)
            if not title or len(title) < 10:
                continue

            href = a_tag.get('href', '').strip()
            if not href:
                continue

            if href.startswith('http://') or href.startswith('https://'):
                full_url = href
            else:
                # Encode non-ASCII chars (like œ) and keep valid URL punctuation.
                safe_href = quote(href, safe='/-_.~')
                full_url = urljoin('https://www.ilboursa.com/marches/', safe_href)

            articles.append({
                'ticker':       ticker,
                'isin_code':    None,
                'title':        title,
                'content':      '',
                'source':       'ilboursa.com',
                'url':          full_url,
                'published_at': str(published),
                'language':     'fr',
                'credibility':  0.90,
            })

    except Exception as e:
        if debug:
            print(f"  Exception for {ticker}: {e}")

    return articles


# ── TEST with AMEN BANK ────────────────────────────────────────────────────
print("Testing parser with AMEN BANK...")
test_articles = parse_ilboursa_page('AMEN BANK', 'AB', max_articles=5, debug=True)

print(f"Articles found: {len(test_articles)}")
print()
for a in test_articles:
    print(f"  Title : {a['title']}")
    print(f"  Date  : {a['published_at']}")
    print(f"  URL   : {a['url']}")
    print()

# Do not run Cell 5 until you confirm Cell 4 output is correct.


Testing parser with AMEN BANK...
  Found 5 date spans for AMEN BANK
Articles found: 5

  Title : Mise en œuvre d'un nouveau contrat de liquidité entre Amen Bank et Amen Invest
  Date  : 2026-02-18
  URL   : https://www.ilboursa.com/marches/mise-en-%C5%93uvre-d-un-nouveau-contrat-de-liquidite-entre-amen-bank-et-amen-invest_59888

  Title : Amen Bank publie son Reporting ESG
  Date  : 2026-02-12
  URL   : https://www.ilboursa.com/marches/amen-bank-publie-son-reporting-esg_59754

  Title : Amen Bank réunit experts et entreprises autour de l'intelligence Artificielle
  Date  : 2026-02-10
  URL   : https://www.ilboursa.com/marches/amen-bank-reunit-experts-et-entreprises-autour-de-l-intelligence-artificielle_59675

  Title : Amen Bank réunit experts et entreprises autour de l'intelligence Artificielle
  Date  : 2026-01-26
  URL   : https://www.ilboursa.com/marches/amen-bank-reunit-experts-et-entreprises-autour-de-l-intelligence-artificielle_59286

  Title : Amen Bank annonce un PNB de 590 mi

In [17]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Verify all ticker codes                        ║
# ╚══════════════════════════════════════════════════════════╝
#
# Now that the parser works, verify all 69 codes.
# Codes that return 0 articles need to be corrected manually.
 
def verify_ticker_code(ticker: str, code: str) -> dict:
    """Quick check: how many articles does this ticker/code return?"""
    articles = parse_ilboursa_page(ticker, code, max_articles=3)
    return {
        'ticker':   ticker,
        'code':     code,
        'articles': len(articles),
        'ok':       len(articles) > 0,
    }
 
print("Verifying all ticker codes (takes ~2 minutes)...")
print(f"{'Ticker':<30} {'Code':<12} {'Articles':<10} {'OK?'}")
print("-" * 60)
 
failed = []
 
for ticker, code in ILBOURSA_TICKERS.items():
    result = verify_ticker_code(ticker, code)
    icon   = "✅" if result['ok'] else "❌"
    print(f"{ticker:<30} {code:<12} {result['articles']:<10} {icon}")
    if not result['ok']:
        failed.append({'ticker': ticker, 'code': code})
    time.sleep(1.5)
 
print()
print(f"Working: {len(ILBOURSA_TICKERS) - len(failed)} / {len(ILBOURSA_TICKERS)}")
 
if failed:
    print()
    print("NEED MANUAL FIX — visit these URLs and find the correct s= code:")
    for f in failed:
        url = f"https://www.ilboursa.com/marches/news_valeur?s={f['code']}"
        print(f"  {f['ticker']:<30} current={f['code']}")
        print(f"    {url}")
 

Verifying all ticker codes (takes ~2 minutes)...
Ticker                         Code         Articles   OK?
------------------------------------------------------------
AMEN BANK                      AB           3          ✅
BIAT                           BIAT         3          ✅
ATTIJARI BANK                  TJARI        3          ✅
BH                             BH           3          ✅
BNA                            BNA          3          ✅
BT                             BT           3          ✅
STB                            STB          3          ✅
UIB                            UIB          3          ✅
ATB                            ATB          3          ✅
UBCI                           UBCI         3          ✅
WIFACK INT BANK                WIFAK        3          ✅
STAR                           STAR         3          ✅
ASTREE                         AST          3          ✅
BH ASSURANCE                   BHASS        3          ✅
TUNIS RE                       TR

In [18]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Enrich articles with ISIN codes from DB        ║
# ╚══════════════════════════════════════════════════════════╝
 
def enrich_with_isin(articles: list) -> list:
    """
    Add isin_code to each article by looking up the ticker in company_metadata.
    This links articles to price data in the database.
    """
    if not articles:
        return articles
 
    tickers = list(set(a['ticker'] for a in articles))
 
    conn = get_conn()
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(
                'SELECT ticker, isin_code FROM company_metadata WHERE ticker = ANY(%s)',
                (tickers,)
            )
            isin_map = {row['ticker']: row['isin_code'] for row in cur.fetchall()}
    finally:
        conn.close()
 
    for a in articles:
        a['isin_code'] = isin_map.get(a['ticker'])
 
    return articles
 
 

In [19]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Insert articles into DB                        ║
# ╚══════════════════════════════════════════════════════════╝
#
# ON CONFLICT (url) DO NOTHING:
# If the same URL is scraped twice, PostgreSQL silently skips it.
# This makes the scraper safe to run multiple times.
# cur.rowcount == 1 means a genuinely new row was inserted.
# cur.rowcount == 0 means it was a duplicate and was skipped.
 
def insert_articles(articles: list) -> int:
    """Insert scraped articles. Returns count of NEW articles inserted."""
    if not articles:
        return 0
 
    conn = get_conn()
    inserted = 0
 
    try:
        with conn.cursor() as cur:
            for a in articles:
                try:
                    cur.execute('''
                        INSERT INTO news_articles
                            (ticker, isin_code, title, content, source,
                             url, published_at, language, credibility)
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                        ON CONFLICT (url) DO NOTHING
                    ''', (
                        a.get('ticker'),
                        a.get('isin_code'),
                        a.get('title', ''),
                        a.get('content', ''),
                        a.get('source', ''),
                        a.get('url', ''),
                        a.get('published_at'),
                        a.get('language', 'fr'),
                        a.get('credibility', 0.90),
                    ))
                    if cur.rowcount == 1:
                        inserted += 1
                except Exception as e:
                    print(f"    Skipped one article: {e}")
                    continue
 
        conn.commit()
    finally:
        conn.close()
 
    return inserted
 
 

In [20]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Bulk scrape all stocks                         ║
# ╚══════════════════════════════════════════════════════════╝
#
# Loops over all stocks, scrapes, enriches, inserts.
# Sleeps 2 seconds between each stock.
# Sleeps 30 seconds every 20 stocks to avoid rate limiting.
 
def run_full_scrape(max_articles_per_stock: int = 25):
    total_scraped  = 0
    total_inserted = 0
    zero_stocks    = []
 
    print("=" * 60)
    print(f"Scraping {len(ILBOURSA_TICKERS)} stocks from ilboursa.com")
    print(f"Estimated time: ~{len(ILBOURSA_TICKERS) * 2 // 60} minutes")
    print("=" * 60)
 
    for i, (ticker, code) in enumerate(ILBOURSA_TICKERS.items(), 1):
        print(f"[{i:2d}/{len(ILBOURSA_TICKERS)}] {ticker:<30}", end=" ")
 
        articles = parse_ilboursa_page(ticker, code, max_articles=max_articles_per_stock)
 
        if not articles:
            print("→ 0 articles")
            zero_stocks.append(ticker)
            time.sleep(1)
            continue
 
        articles   = enrich_with_isin(articles)
        inserted   = insert_articles(articles)
        total_scraped  += len(articles)
        total_inserted += inserted
 
        print(f"→ scraped={len(articles)}, new inserted={inserted}")
 
        time.sleep(2)
 
        # Longer pause every 20 stocks
        if i % 20 == 0:
            print("  [30s pause to avoid rate limiting...]")
            time.sleep(30)
 
    print()
    print("=" * 60)
    print("DONE")
    print(f"  Total scraped:  {total_scraped}")
    print(f"  New inserted:   {total_inserted}")
    print(f"  Zero articles:  {len(zero_stocks)}")
    if zero_stocks:
        print()
        print("  Stocks with 0 articles (codes need fixing):")
        for s in zero_stocks:
            print(f"    {s:<30} code={ILBOURSA_TICKERS[s]}")
    print("=" * 60)
 
    return total_inserted
 
 
# Run it only after Cell 4 and Cell 5 confirm the parser works
total = run_full_scrape(max_articles_per_stock=25)
 
 

Scraping 68 stocks from ilboursa.com
Estimated time: ~2 minutes
[ 1/68] AMEN BANK                      → 0 articles
[ 2/68] BIAT                           → scraped=20, new inserted=20
[ 3/68] ATTIJARI BANK                  → scraped=20, new inserted=20
[ 4/68] BH                             → scraped=20, new inserted=20
[ 5/68] BNA                            → scraped=20, new inserted=20
[ 6/68] BT                             → scraped=20, new inserted=20
[ 7/68] STB                            → scraped=20, new inserted=20
[ 8/68] UIB                            → scraped=20, new inserted=20
[ 9/68] ATB                            → scraped=20, new inserted=20
[10/68] UBCI                           → scraped=20, new inserted=20
[11/68] WIFACK INT BANK                → scraped=20, new inserted=20
[12/68] STAR                           → scraped=20, new inserted=20
[13/68] ASTREE                         → scraped=20, new inserted=20
[14/68] BH ASSURANCE                   → scraped=20, new

In [22]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL — Re-scrape AMEN BANK only                         ║
# ╚══════════════════════════════════════════════════════════╝

# Wait 10 seconds first so the session is fully warmed up
print("Waiting 10 seconds before retrying AMEN BANK...")
time.sleep(10)

articles = parse_ilboursa_page('AMEN BANK', 'AB', max_articles=25, debug=True)

if articles:
    articles = enrich_with_isin(articles)
    inserted = insert_articles(articles)
    print(f"Success — scraped={len(articles)}, new inserted={inserted}")
else:
    print("Still 0 — run this cell again in 2 minutes")

Waiting 10 seconds before retrying AMEN BANK...
  Found 20 date spans for AMEN BANK
Success — scraped=20, new inserted=20


In [23]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Verify results                                 ║
# ╚══════════════════════════════════════════════════════════╝
 
conn = get_conn()
 
df = pd.read_sql('''
    SELECT
        ticker,
        COUNT(*)                AS articles,
        MIN(published_at)::text AS oldest,
        MAX(published_at)::text AS newest
    FROM news_articles
    WHERE source = 'ilboursa.com'
    GROUP BY ticker
    ORDER BY articles DESC
''', conn)
 
print(f"Stocks with articles: {len(df)}")
print()
print(df.to_string(index=False))
 
# Stocks still at zero (will be covered by other sources in next sessions)
df_zero = pd.read_sql('''
    SELECT cm.ticker
    FROM company_metadata cm
    LEFT JOIN news_articles na ON cm.ticker = na.ticker
    WHERE na.ticker IS NULL
    ORDER BY cm.ticker
''', conn)
 
print()
print(f"Stocks with 0 articles (for next scraper): {len(df_zero)}")
print(df_zero['ticker'].tolist())
 
conn.close()
 

Stocks with articles: 68

            ticker  articles     oldest     newest
    TELNET HOLDING        20 2024-01-22 2026-01-16
    MODERN LEASING        20 2022-03-31 2026-03-11
              MPBS        20 2022-10-21 2025-09-01
              SFBT        20 2024-10-20 2026-01-31
              BIAT        20 2025-04-25 2026-03-04
   AIR LIQUDE TSIE        20 2019-01-21 2025-10-07
        GIF-FILTER        20 2018-01-05 2024-11-12
            SIMPAR        20 2017-08-30 2023-04-28
              SITS        20 2020-11-25 2025-09-12
             SIAME        20 2023-09-12 2026-03-04
          MONOPRIX        20 2023-10-06 2026-01-22
        BEST LEASE        20 2017-03-16 2025-03-21
            UNIMED        20 2024-01-25 2026-01-20
    DELICE HOLDING        20 2022-06-27 2026-01-21
          TUNIS RE        20 2024-05-06 2026-02-23
             SOPAT        20 2020-10-13 2024-11-22
CIMENTS DE BIZERTE        20 2022-09-26 2026-01-29
  ONE TECH HOLDING        20 2024-10-22 2026-02-25
     

C:\Users\Negza\AppData\Local\Temp\ipykernel_1404\3891850263.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('''
C:\Users\Negza\AppData\Local\Temp\ipykernel_1404\3891850263.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_zero = pd.read_sql('''
